In [ ]:
"""
ETL - FASE 1: EXTRAÇÃO (Extract)
Trabalho de Conclusão de Curso - IFES
Descrição: Este script lê múltiplos ficheiros PDF contendo os editais
de aprovação do IFES e consolida-os num único ficheiro CSV bruto.
"""

import pandas as pd
import tabula
import os
import re

# --- 1. CONFIGURAÇÃO DE DIRETÓRIOS ---
# Altere para o caminho onde estão as suas pastas de PDFs
caminho_base = r"./dados"
pastas_de_dados = ["PDFs_IFES_2025", "PDFs_IFES_2024", "PDFs_IFES_2023", "PDFs_IFES_2022"]
arquivo_saida = os.path.join(caminho_base, "dados_ifes_brutos_consolidados.csv")
lista_dfs = []

print("--- INICIANDO SCRIPT DE EXTRAÇÃO (ETL - FASE 1) ---")

# --- 2. PROCESSAMENTO DE EXTRAÇÃO ---
for nome_pasta in pastas_de_dados:
    caminho_pasta_ano = os.path.join(caminho_base, nome_pasta)

    if not os.path.exists(caminho_pasta_ano):
        print(f"[AVISO] Pasta não encontrada, pulando: {nome_pasta}")
        continue

    try:
        # Extrai o ano do nome da pasta (ex: 2024)
        ano = re.search(r'\d{4}', nome_pasta).group()
        print(f"\n--- Processando pasta do ANO: {ano} ---")
    except AttributeError:
        continue

    for nome_arquivo in os.listdir(caminho_pasta_ano):
        if nome_arquivo.endswith(".pdf"):
            caminho_pdf = os.path.join(caminho_pasta_ano, nome_arquivo)

            # Extração de Metadados do nome do ficheiro
            partes_nome = nome_arquivo.replace(".pdf", "").split("-")
            campus = partes_nome[0].title()

            try:
                idx_integrado_em = partes_nome.index("em")
                curso = " ".join(partes_nome[idx_integrado_em + 1:]).title()
                # Limpeza de lixo no nome do curso
                curso = re.sub(r' (Matutino|Vespertino|Integral|Noturno|Resultado|Geral).*$', '', curso, flags=re.IGNORECASE)
            except ValueError:
                curso = "Nao Identificado"

            print(f"Lendo: {nome_arquivo} | Campus: {campus} | Curso: {curso}")

            try:
                # Leitura do PDF via Tabula
                dfs_paginas = tabula.read_pdf(
                    caminho_pdf,
                    pages='all',
                    stream=True,
                    encoding='latin-1',
                    pandas_options={'header': None}
                )

                if not dfs_paginas:
                    continue

                df_temp = pd.concat([df for df in dfs_paginas if not df.empty], ignore_index=True)

                # Normalização das colunas
                df_temp.columns = [f"COL_{i}" for i in range(len(df_temp.columns))]
                df_temp['ANO'] = ano
                df_temp['CAMPUS'] = campus
                df_temp['CURSO'] = curso

                lista_dfs.append(df_temp)

            except Exception as e:
                print(f"    [ERRO] Falha ao processar {nome_arquivo}: {e}")

# --- 3. EXPORTAÇÃO (Save) ---
if lista_dfs:
    print("\nConsolidando os dados num único DataFrame...")
    df_final = pd.concat(lista_dfs, ignore_index=True)
    df_final.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')
    print(f"[SUCESSO] Ficheiro bruto gerado: {arquivo_saida}")
else:
    print("[AVISO] Nenhum dado foi extraído.")

In [ ]:
"""
ETL - FASE 2: TRANSFORMAÇÃO (Transform)
Trabalho de Conclusão de Curso - IFES
Descrição: Este script limpa a base de dados bruta, filtra apenas os candidatos
"Aprovados" ou "Classificados", e calcula a Tabela de Ouro (KPIs de Corte).
"""

import pandas as pd
import numpy as np
import os

# --- 1. CONFIGURAÇÃO ---
caminho_base = r"./dados"
arquivo_csv_mestre = os.path.join(caminho_base, "dados_ifes_brutos_consolidados.csv")
saida_corte_geral = os.path.join(caminho_base, "TENDENCIA_NOTAS_CORTE_LIMPO.csv")

print("--- INICIANDO LIMPEZA E CÁLCULO DE KPIs (ETL - FASE 2) ---")

try:
    # low_memory=False para evitar alertas de tipagem do Pandas
    df = pd.read_csv(arquivo_csv_mestre, low_memory=False)
except FileNotFoundError:
    print(f"[ERRO] Arquivo não encontrado: {arquivo_csv_mestre}")
    exit()

# --- 2. DATA WRANGLING E FILTRAGEM ---
# Filtrar apenas alunos aprovados para descobrir a nota do último colocado
status_limpos = ['Aprovado', 'Classificado']

# Assegurar que a coluna de RESULTADO existe e tem o nome correto (Ajuste conforme o seu dataset)
col_resultado = 'RESULTADO' if 'RESULTADO' in df.columns else df.columns[-1]
col_nota = 'NOTA_FINAL' if 'NOTA_FINAL' in df.columns else df.columns[-2]

df_clean = df[df[col_resultado].isin(status_limpos)].copy()
df_clean[col_nota] = pd.to_numeric(df_clean[col_nota], errors='coerce')
df_clean = df_clean.dropna(subset=[col_nota])

print(f"Total de candidatos aprovados/classificados processados: {len(df_clean)}")

# --- 3. CÁLCULO DOS KPIs (TABELA DE OURO) ---
print("Calculando Notas de Corte (Mínimo) e Desempenho Percentual...")

df_kpis = df_clean.groupby(['ANO', 'CAMPUS', 'CURSO'])[col_nota].agg(
    NOTA_MAXIMA='max',
    PONTO_DE_CORTE='min', # Nota do último aprovado
    MEDIA_CURSO='mean'
).reset_index()

# Normalização de Escala (As provas do IFES variaram entre 500 e 400 pontos nos últimos anos)
# Para uniformizar, criamos um índice de desempenho percentual baseado na prova daquele ano.
def calcular_percentual(row):
    # Se o ano for menor que 2024, a prova valia 500. Senão, 400.
    teto_prova = 500.0 if int(row['ANO']) < 2024 else 400.0
    return row['PONTO_DE_CORTE'] / teto_prova

df_kpis['DESEMPENHO_CORTE'] = df_kpis.apply(calcular_percentual, axis=1)

# --- 4. EXPORTAÇÃO ---
# Ordenar para facilitar a leitura visual
df_kpis = df_kpis.sort_values(by=['ANO', 'CAMPUS', 'CURSO'], ascending=[False, True, True])
df_kpis.to_csv(saida_corte_geral, index=False, encoding='utf-8-sig')

print(f"[SUCESSO] Tabela de Ouro gerada com sucesso: {saida_corte_geral}")

In [ ]:
"""
ETL - FASE 3: CARGA (Load)
Trabalho de Conclusão de Curso - IFES
Descrição: Conecta-se ao banco de dados relacional na nuvem (Supabase)
e faz a inserção (ou atualização) da Tabela de Ouro gerada na Fase 2.
"""

import pandas as pd
import psycopg2
from psycopg2 import extras
import os

# --- 1. CREDENCIAIS ---
# ATENÇÃO: Nunca suba esta senha para o GitHub público! Use variáveis de ambiente num projeto real.
DB_URI = "postgresql://postgres:<SUBA_SENHA_AQUI>@db.efandhnqvcukmvfsoeoz.supabase.co:5432/postgres"

caminho_base = r"./dados"
arquivo_csv = os.path.join(caminho_base, "TENDENCIA_NOTAS_CORTE_LIMPO.csv")

print("--- INICIANDO UPLOAD PARA O SUPABASE (ETL - FASE 3) ---")

try:
    df = pd.read_csv(arquivo_csv)

    # Remover duplicatas exatas para não violar a Primary Key do Banco de Dados
    df = df.drop_duplicates(subset=['ANO', 'CAMPUS', 'CURSO'], keep='first')
    print(f"CSV preparado para envio: {len(df)} registos únicos.")

    # Conectar ao Postgres do Supabase
    conn = psycopg2.connect(DB_URI)
    cur = conn.cursor()

    # Recriar Tabela e Configurar Segurança (RLS - Row Level Security)
    print("Configurando esquema da tabela 'kpis_ifes'...")
    cur.execute("""
        DROP TABLE IF EXISTS kpis_ifes;

        CREATE TABLE kpis_ifes (
            ano INT,
            campus TEXT,
            curso TEXT,
            nota_maxima REAL,
            ponto_de_corte REAL,
            desempenho_corte REAL,
            PRIMARY KEY (ano, campus, curso)
        );

        -- Permite que a aplicação Web leia os dados anonimamente
        ALTER TABLE kpis_ifes ENABLE ROW LEVEL SECURITY;
        CREATE POLICY "Permitir leitura publica" ON kpis_ifes FOR SELECT USING (true);
    """)
    conn.commit()

    # Preparar as tuplas para o envio em lote (Batch Insert)
    registros = []
    for _, row in df.iterrows():
        registros.append((
            int(row['ANO']),
            str(row['CAMPUS']),
            str(row['CURSO']),
            float(row['NOTA_MAXIMA']),
            float(row['PONTO_DE_CORTE']),
            float(row['DESEMPENHO_CORTE'])
        ))

    # Query de Inserção com tratamento de conflito (Upsert)
    query = """
        INSERT INTO kpis_ifes (ano, campus, curso, nota_maxima, ponto_de_corte, desempenho_corte)
        VALUES %s
        ON CONFLICT (ano, campus, curso)
        DO UPDATE SET
            nota_maxima = EXCLUDED.nota_maxima,
            ponto_de_corte = EXCLUDED.ponto_de_corte,
            desempenho_corte = EXCLUDED.desempenho_corte
    """

    print("A enviar dados...")
    extras.execute_values(cur, query, registros)
    conn.commit()

    print("[SUCESSO ABSOLUTO] Os dados estão online e prontos para a Aplicação Web!")

except Exception as e:
    print(f"[ERRO CRÍTICO] Falha no processo: {e}")
finally:
    if 'conn' in locals() and conn:
        cur.close()
        conn.close()
        print("Conexão à nuvem encerrada com segurança.")